### 멀티모달 RAG
- 01.multimodal 에서 추출한 텍스트와 이미지요약본을 VectorDB에 임베딩하고 

    사용자의 질문에 맞춰 관련 컨텍스트를 검색(Retrival)한뒤 LLM을 이용해서 최종 답변을 생성(Generation)

In [1]:
import os
import fitz
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions

In [2]:
# 환경변수 로드 및 openai 클라이언트 생성
load_dotenv(override=True)
client = OpenAI()

In [20]:
# 벡터 Db 초기화
chroma_client = chromadb.PersistentClient(path='./chroma_db')
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv('OPENAI_API_KEY'),
    model_name = 'text-embedding-3-small'
)
if chroma_client.count_collections() != 0:
    chroma_client.delete_collection('multimodal_rag')
collection = chroma_client.get_or_create_collection(name='multimodal_rag',embedding_function=openai_ef)
print(f'설정완료 현재 컬렉션의 크기 : {collection.count()}')

설정완료 현재 컬렉션의 크기 : 0


### 데이터 임베딩(텍스트 + 이미지 요약본)
- pdf의 텍스트와 이미지 요약본을 모두 VectorDB에 삽입(동일한 벡터공간에 텍스트와 이미지 요약본이 함께 존재)

In [21]:
import base64
if collection.count() == 0:
    pdf_path = 'sample_paper.pdf'
    doc = fitz.open(pdf_path)
    documents = []
    metatdatas = []    
    ids = []
    
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        # 텍스트 추출
        text = page.get_text()
        if text.strip():
            documents.append(text)
            metatdatas.append({'page':page_num+1,'type':'text'})
            ids.append(f'text_page_{page_num+1}')
        # 이미지 요약 후 추출
        image_list = page.get_images(full=True)
        if image_list:
            xref = image_list[0][0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image['image']
            ext = base_image['ext']
            if ext.lower() == 'jb2': ext = 'jpeg'
            mime_type = f'image/{ext}' if ext.lower() != "jpg" else 'image/jpeg'
            base64_image = base64.b64encode(image_bytes).decode('utf-8')
            try:
                response = client.chat.completions.create(
                    model = 'gpt-5.4-nano',
                    messages=[
                        {'role':'user', 
                        'content': [
                            {'type':'text', 'text':'이 이미지에 대한 상세한 설명을 작성해 주세요. 전문적이고 구체적으로 분석해야 합니다.'},
                            {'type':'image_url', 'image_url':{
                                'url':f'data:{mime_type};base64,{base64_image}'
                                }
                            }
                        ]
                        }
                    ],
                    max_completion_tokens=300
                )
                summary = response.choices[0].message.content
                documents.append(summary)
                metatdatas.append({
                    'type':'image_summary','page':page_num+1
                })
                ids.append(f'image_summary_page_{page_num+1}')
            except Exception as e:
                pass
    if documents:
        collection.add(documents=documents,metadatas=metatdatas,ids=ids)
        print('VectorDB에 데이터 임베딩 및 저장 완료')
else:
    print(f'vectorDB에 이미 {collection.count()}개 데이터가 존재합니다')


VectorDB에 데이터 임베딩 및 저장 완료


### 질문을 통해 RAG 검색하기

In [23]:
query = '논문에 나온 트랜스포커 아키텍처 다이어그램(인코더-디코더 그림)에 대해 설명해주세요'
results = collection.query(
    query_texts=[query],
    n_results=3
)
results

{'ids': [['image_summary_page_3', 'image_summary_page_4', 'text_page_12']],
 'embeddings': None,
 'documents': [['아래 이미지는 **Transformer(트랜스포머) 시퀀스-투-시퀀스 아키텍처**의 한 예로, **인코더(좌측)**–**디코더(우측)**로 구성된 전체 흐름을 한 장의 다이어그램으로 설명합니다. 핵심적으로는 *임베딩 + 위치 인코딩 → 인코더 층 반복(좌측) → 디코더 층 반복(우측, 마스크드 셀프어텐션 포함) → 선형/소프트맥스 출력 확률*의 구조가 시각화되어 있습니다.\n\n---\n\n## 1) 전체 레이아웃(기능 블록의 배치)\n- **좌측(Encoder / 인코더)**: 반복 블록이 **N×**로 표시되어 있으며, 입력 문장을 문맥적으로 인코딩합니다.\n- **우측(Decoder / 디코더)**: 역시 **N×**로 반복되며, 출력 문장을 생성하기 위한 조건부 처리를 수행합니다.\n- 중앙 상단: 디코더 최상단에서 나온 표현이 **Linear → Softmax**를 거쳐 **“Output Probabilities(출력 확률)”**로 변환됩니다.\n\n또한 좌우 각각에 **Input Embedding**, **Output Embedding**이 있고, 위치 정보를 주기 위한 **Positional Encoding**이 임',
   '아래 이미지는 **Transformer(트랜스포머) 구조의 한 어텐션 헤드(또는 유사한 attention 계산 흐름)**에서 사용되는 연산을 도식화한 것입니다. 전체적으로 입력 텐서가 위(상단)로 흐르면서 **Q/K/V를 기반으로 attention 점수 → 마스킹 → 정규화(softmax) → 가중합**을 수행하는 과정을 블록 형태로 표현합니다.\n\n---\n\n## 1) 흐름(좌→우가 아니라 위→아래/위→아래의 연산 순서)\n이미지는 블록들이 **위쪽 화살표**로 연결되어 있어, 데이터가 아래에서 위로(또는 반대로 읽더라